# 06 — the EXPLAIN comparison workbook

Builds `reports/api-explain-<case>-<stamp>.xlsx`, **one file per case**.

The markdown reports are the narrative; this is the instrument. 1,414 statements
across three identities cannot be compared by scrolling — a plan rendered beside
its statement is right for reading one API and useless for asking which
statements differ.

Design and the reasoning behind each sheet: `PLAN-explain-workbook.md`.

**Touches no database and no cluster.** Two files in, one workbook out.

## Sheets

| sheet | one row per | answers |
|---|---|---|
| `Summary` | API | which operations seq-scan, and on what |
| `Statements` | statement occurrence | what one API issues, in order |
| `Distinct` | (SQL, params) pair | what the sweep planned, with weight |
| `Cost` | (SQL, params) pair | **which statement is expensive, and next to what** |
| `Shapes` | logical statement | near-duplicates collapsed by predicate SET |
| `Unplanned` | refusal | redacted (permanent) vs refused (recoverable) |
| `Schema` | live index | what exists, and what upstream defines |
| `Provenance` | — | sources, volumes, and what NOT to do with the numbers |

## The `Cost` sheet — the comparison, without a hypothetical index

`rows_scanned` is what the plan READS from its table: the whole table for a
Seq Scan, the `Index Cond` match for an index node. It is **not** `plan_rows`,
which on a Seq Scan is what survives the `Filter` and reads `1` on a
60,815-row table.

`indexed_baseline` is **not** a what-if. No index was created for this analysis
and the schema is the plain upstream one. It is the cheapest *indexed* access to
the **same table** measured in the **same sweep** — `grant_records` is reached
both ways here, sequentially for the grantee direction and through
`grant_records_pkey` for the securable one. `rows_x` and `cost_x` are that
comparison; a table only ever seq-scanned gets no baseline and no ratio rather
than a borrowed one.

`rows_x` and `cost_x` disagree by two orders of magnitude and both are right:
`rows_x` is how much **wider** the read is, `cost_x` is what the planner thinks
that width **costs** — much less than proportional, because a sequential read is
far cheaper per row than the same number of random index descents. Quote both or
neither.

`cost_units` is `total_cost × occurrences` and `cost_share` its fraction of the
case. Planner units, not milliseconds, and a within-case ranking only.

## Read `Provenance` before doing arithmetic

`total_cost` and `plan_rows` are planner **estimates**. Two sweeps 25 minutes
apart differed in 11 of 264 `plan_rows` while **0 of 264** differed in plan
*shape*. Sort and compare shape; do not average, sum or trend the estimates.
`duration_ms` is from the capture, not the EXPLAIN — plain `EXPLAIN` does not
execute.


In [ ]:
# --- cell 1: bootstrap ------------------------------------------------------
import gc
import pathlib
import subprocess
import sys

HERE = pathlib.Path.cwd()
REPORTS = HERE / "reports"
RUNS = HERE / "runs"
print(f"reports: {REPORTS}")


## Inputs — set these by hand

One entry per case. Pair them deliberately: a report and a run from different
sweeps still join on most statements and are wrong about all of them. The
generator refuses a run whose `case` or `matrix` disagrees, but it cannot catch
two files that are merely from different *drives* of the same case.


In [ ]:
# --- cell 2: INPUTS ---------------------------------------------------------
BUILD = {
    "unauthorized": {
        "matrix": "doc-api-sql-matrix-unauthorized-20260902-171852.md",
        "explain": "apiexplain-unauthorized-20260903-110148.json",
    },
    "authorized": {
        "matrix": "doc-api-sql-matrix-authorized-20260902-172636.md",
        "explain": "apiexplain-authorized-20260903-110149.json",
    },
    "admin": {
        "matrix": "doc-api-sql-matrix-admin-20260903-094137.md",
        "explain": "apiexplain-admin-20260903-110152.json",
    },
}
for case, io in BUILD.items():
    assert (REPORTS / io["matrix"]).exists(), f"{case}: no {io['matrix']}"
    assert (RUNS / io["explain"]).exists(), f"{case}: no {io['explain']}"
    print(f"{case:<13} {io['matrix']}\n{'':<13} + {io['explain']}")


In [ ]:
# --- cell 3: build ----------------------------------------------------------
BUILT = {}
for case, io in BUILD.items():
    print(f"\n=== {case} ===")
    p = subprocess.run(
        [sys.executable, "render_explain_workbook.py", "--case", case,
         "--matrix", f"reports/{io['matrix']}",
         "--explain", f"runs/{io['explain']}"],
        cwd=str(HERE), capture_output=True, text=True,
    )
    print(p.stdout.rstrip() or p.stderr.rstrip())
    assert p.returncode == 0, f"{case}: generator exited {p.returncode}"
    BUILT[case] = p.stdout.splitlines()[0].split("-> ")[-1].strip()


## Check the counts against the reports

The generator reads the same files the reports were rendered from, so these must
reconcile. A silent mismatch means the wrong pair was named above.


In [ ]:
# --- cell 4: reconcile ------------------------------------------------------
import json
import re

for case, io in BUILD.items():
    text = (REPORTS / io["matrix"]).read_text(encoding="utf-8")
    run = json.loads((RUNS / io["explain"]).read_text())
    print(f"{case:<13} report statements={len(re.findall(r'```sql', text)):<5} "
          f"run pairs={run['pairs']:<5} index={run['index_state']:<8} "
          f"grant_records={(run.get('volume') or {}).get('grant_records'):,}")
    #: 43 in every case, in every identity -- the surface is fixed and only the
    #: outcomes change. A different number here means a partial drive.
    assert len(run["per_api"]) == 43, f"{case}: {len(run['per_api'])} APIs, expected 43"


In [ ]:
# --- cell 5: what was written -----------------------------------------------
for case, name in BUILT.items():
    f = REPORTS / name
    print(f"{case:<13} {name}  ({f.stat().st_size / 1024:.0f} KB)")
print("\nOpen Provenance first — it names the sources and says what the "
      "estimates may not be used for.")


In [ ]:
# --- cell 5b: the headline, without opening Excel ---------------------------
#: The Cost sheet's top rows. Printed here because the finding is one number
#: per case and asking someone to open three workbooks to see it loses it.
from openpyxl import load_workbook

for case, name in BUILT.items():
    ws = load_workbook(REPORTS / name, read_only=True)["Cost"]
    it = ws.iter_rows(values_only=True)
    h = {k: i for i, k in enumerate(next(it))}
    print(f"\n=== {case} ===")
    print(f"{'table':<15}{'verb':<8}{'occ':>5}{'rows_scanned':>14}"
          f"{'rows_x':>9}{'cost_x':>8}{'share':>8}  plan")
    for r in list(it)[:5]:
        print(f"{r[h['table']]:<15}{r[h['verb']]:<8}{r[h['occurrences']]:>5}"
              f"{r[h['rows_scanned']]:>14,}{r[h['rows_x']]:>9,}"
              f"{r[h['cost_x']]:>8}{r[h['cost_share']]:>7.1%}  {r[h['plan']]}")
    ws.parent.close()


In [ ]:
# --- cell 6: teardown -------------------------------------------------------
for _n in ("BUILT",):
    if _n in dir():
        del globals()[_n]
gc.collect()
print("done")
